# Lab 1 SOLUTIONS: Basic RAG with LlamaParse
## CCI Session 6

In [ ]:
!pip install -q llama-parse llama-index langchain langchain-openai langchain-community langchain-chroma chromadb pypdf

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')

In [ ]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = '/content/WT.pdf'

## Cell 1 — Parse with LlamaParse

In [ ]:
from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=os.environ['LLAMA_CLOUD_API_KEY'],
    result_type='markdown',
    verbose=True,
)
documents = parser.load_data(PDF_PATH)

print(f'Number of documents: {len(documents)}')
print(documents[0].text[:1500])

## Cell 2 — Naive PyPDF Comparison

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
naive_docs = PyPDFLoader(PDF_PATH).load()
print(f'Pages (PyPDF): {len(naive_docs)}')
print('--- PyPDF (jumbled) ---')
print(naive_docs[5].page_content[:1500])
print('\n--- LlamaParse (markdown) ---')
print(documents[0].text[1500:3000])

## Cell 3 — Chunk

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in documents]

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(lc_docs)

print(f'Chunks: {len(chunks)}')
print(chunks[10].page_content[:600])

## Cell 4 — Embeddings

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## Cell 5 — Chroma Vector Store

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_wt',
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

## Cell 6 — RAG Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a clinical assistant. Answer the question using ONLY the provided context. If unknown, say so.\n\nContext:\n{context}'),
    ('human', '{input}')
])

doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

for q in [
    'What is the standard treatment for Stage III favorable histology Wilms tumor?',
    'What are the most common side effects of vincristine?'
]:
    res = rag_chain.invoke({'input': q})
    print(f'\nQ: {q}\nA: {res["answer"]}')

## Cell 7 — Citations

In [ ]:
q1 = 'What is the standard treatment for Stage III favorable histology Wilms tumor?'
res = rag_chain.invoke({'input': q1})
for i, doc in enumerate(res['context']):
    print(f'--- Chunk {i+1} ---')
    print(doc.page_content[:300])
    print()

## Stretch — different chunk sizes

In [ ]:
for cs in [500, 2000]:
    sp = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=int(cs*0.2))
    ch = sp.split_documents(lc_docs)
    print(f'chunk_size={cs} → {len(ch)} chunks')